# 10 - Stateful Orchestration with LangGraph

## Scenario: Northstar State Management

When Northstar support agents (AI or human) investigate an issue, they maintain context across multiple interactions. In a basic `while` loop (like we built in earlier modules), state is just a list of messages. This becomes unmanageable quickly when we need to track structured data like `incident_id`, `current_hypothesis`, or `escalated_to_human`.

In this module, we will use **LangGraph** to build a stateful agent. LangGraph treats the agent as a graph of nodes and edges, passing a strictly defined `State` object between them.

In [1]:
import json
from typing import TypedDict, Annotated, Sequence
import operator
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage

# 1. Define the State
# The state is a TypedDict that flows through the graph.
# 'messages' uses an operator.add reducer to append new messages instead of overwriting them.
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]
    incident_id: str
    confidence: float
    escalated: bool


## 1. Defining Nodes

In LangGraph, a node is just a Python function that takes the current `State`, does some work, and returns a dictionary with state updates. LangGraph merges these updates into the main state.

In [2]:
def node_investigate(state: AgentState):
    print("[Node: Investigate] Analyzing evidence...")
    # Simulated LLM thought process
    if "checkout" in state["messages"][-1].content.lower():
        new_confidence = state.get("confidence", 0.0) + 0.6
        return {
            "incident_id": "INC-9912",
            "confidence": new_confidence,
            "messages": [AIMessage(content="I found an active checkout incident.")]
        }
    return {"messages": [AIMessage(content="I am not sure what the issue is.")]}

def node_escalate(state: AgentState):
    print("[Node: Escalate] Confidence is too low. Escalating to human...")
    return {
        "escalated": True,
        "messages": [AIMessage(content="I have escalated this to the engineering team.")]
    }

def node_resolve(state: AgentState):
    print("[Node: Resolve] Confidence is high. Proposing resolution...")
    return {
        "messages": [AIMessage(content=f"Incident {state['incident_id']} is confirmed. Mitigation in progress.")]
    }


## 2. Defining Conditional Edges (Routing)

Edges determine where the graph goes next. A conditional edge uses a function to inspect the state and return the name of the next node.

In [3]:
def router(state: AgentState) -> str:
    # If confidence > 0.8, we resolve. Else, we escalate.
    if state.get("confidence", 0.0) > 0.8:
        return "resolve"
    return "escalate"


## 3. Assembling the Graph

Now we wire the nodes and edges together into a `StateGraph` and compile it into an executable app.

In [4]:
from langgraph.graph import StateGraph, END

# Initialize graph with our State schema
builder = StateGraph(AgentState)

# Add nodes
builder.add_node("investigate", node_investigate)
builder.add_node("escalate", node_escalate)
builder.add_node("resolve", node_resolve)

# Define the flow
builder.set_entry_point("investigate")

# Add conditional routing from 'investigate'
builder.add_conditional_edges(
    "investigate", 
    router, 
    {
        "resolve": "resolve", 
        "escalate": "escalate"
    }
)

# Both resolve and escalate lead to the end of the graph
builder.add_edge("resolve", END)
builder.add_edge("escalate", END)

# Compile the graph
app = builder.compile()


## 4. Running the Graph

Let's test our stateful orchestration. Notice how the structured state allows us to make routing decisions without relying on fragile prompt engineering (e.g. "If you don't know, output the word ESCALATE").

In [5]:
# Test Case 1: Confidence is initially 0, so it will reach 0.6, which triggers an escalation.
initial_state = {
    "messages": [HumanMessage(content="Checkout is broken!")],
    "confidence": 0.0,
    "escalated": False
}

print("--- Test Case 1: Low Confidence ---")
result_state = app.invoke(initial_state)

print("\nFinal State:")
print(f"Incident ID: {result_state.get('incident_id')}")
print(f"Confidence: {result_state.get('confidence')}")
print(f"Escalated: {result_state.get('escalated')}")
print(f"Final Message: {result_state['messages'][-1].content}\n")


# Test Case 2: Confidence is initially 0.5, so it reaches 1.1, triggering resolution.
initial_state_high = {
    "messages": [HumanMessage(content="Checkout is broken!")],
    "confidence": 0.5,
    "escalated": False
}

print("--- Test Case 2: High Confidence ---")
result_state_high = app.invoke(initial_state_high)

print("\nFinal State:")
print(f"Incident ID: {result_state_high.get('incident_id')}")
print(f"Confidence: {result_state_high.get('confidence')}")
print(f"Escalated: {result_state_high.get('escalated')}")
print(f"Final Message: {result_state_high['messages'][-1].content}\n")


--- Test Case 1: Low Confidence ---
[Node: Investigate] Analyzing evidence...
[Node: Escalate] Confidence is too low. Escalating to human...

Final State:
Incident ID: INC-9912
Confidence: 0.6
Escalated: True
Final Message: I have escalated this to the engineering team.

--- Test Case 2: High Confidence ---
[Node: Investigate] Analyzing evidence...
[Node: Resolve] Confidence is high. Proposing resolution...

Final State:
Incident ID: INC-9912
Confidence: 1.1
Escalated: False
Final Message: Incident INC-9912 is confirmed. Mitigation in progress.



## Watch For

- **Unbound State Growth**: Appending to a list infinitely will break the LLM's context window. Implement message trimming logic.
- **Hidden Transitions**: Implementing complex routing logic directly inside a Node instead of using a `Conditional Edge`.
- **State Overwrites**: Forgetting `operator.add` on a list field, causing a node to accidentally delete the entire message history.

## Checkpoint

**1. Why use a `TypedDict` for LangGraph state instead of just passing a list of messages?**
- A) To force the user to write Python.
- B) To track structured variables (like incident_id) alongside messages, enabling programmatic routing.
- C) It improves LLM generation speed.
- D) It bypasses API limits.

**2. What happens if you define a list in `AgentState` without `Annotated[Sequence, operator.add]`?**
- A) LangGraph will throw a compilation error.
- B) The list will be immutable.
- C) Returning a new list from a Node will overwrite the existing list completely, instead of appending to it.
- D) The LLM will refuse to run.
